In [2]:
import os
import shutil

import kagglehub
import pandas as pd

if not os.path.isdir("nlp-getting-started"):
    path = kagglehub.competition_download("nlp-getting-started")
    shutil.move(path, "nlp-getting-started")

df_train = pd.read_csv("nlp-getting-started/train.csv")
df_test = pd.read_csv("nlp-getting-started/test.csv")

df_train

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [4]:
import re
import gzip
import json
import time
import uuid

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from geopy.geocoders import Nominatim
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def tokenize(doc):
     return re.findall(r"\w+", doc) # TODO: make better tokenizer

class D2VVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, **kwargs):
        self.model = None
        self.hyperparameters = kwargs
    
    def fit(self, X, y=None):
        print(X, type(X))
        self.model = Doc2Vec([TaggedDocument(tokenize(x), [i]) for i, x in enumerate(X)], **self.hyperparameters)
        return self

    def transform(self, X, y=None):
        print(X, type(X))
        return np.vstack([self.model.infer_vector(tokenize(x)) for x in X])

class GeopyLocator(BaseEstimator, TransformerMixin):
    def __init__(self, **kwargs):
        try:
            with gzip.open("geopy_cache.json.gz", "rt") as file:
                self.cache = json.load(file)
        except FileNotFoundError:
            self.cache = {}
            
        self.geolocator = Nominatim(user_agent="ece_6420_final")
    
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        locations = []
        dirty = False
        
        try:
            for x in X:
                if x != x: # NaN
                    locations.append([np.nan, np.nan])
                elif x in self.cache:
                    if self.cache[x] is None:
                        locations.append([np.nan, np.nan])
                    else:
                        locations.append(self.cache[x])
                else:
                    print("Geocoding ", x, "...")
                    location = self.geolocator.geocode(x)
                    dirty = True
                    time.sleep(1) # Respect Nominatim usage policy

                    if location is None:
                        self.cache[x] = None
                        locations.append([np.nan, np.nan])
                    else:
                        self.cache[x] = [location.latitude, location.longitude]
                        locations.append([location.latitude, location.longitude])
                        
            return np.array(locations)
        finally:
            if dirty:
                with gzip.open("geopy_cache.json.gz", "wt") as file:
                    json.dump(self.cache, file)
    
ct = make_column_transformer(
    ("drop", ["id"]),
    (OneHotEncoder(drop=[np.nan]), ["keyword"]),
    (make_pipeline(GeopyLocator(), SimpleImputer(), StandardScaler()), "location"),
    (D2VVectorizer(), "text")
)

ct.fit_transform(df_train)

Geocoding  Nairobi, Kenya ...
Geocoding  Instagram - @heyimginog  ...
Geocoding  304 ...
Geocoding  Switzerland ...
Geocoding  US ...
Geocoding  Somewhere Only We Know ? ...
Geocoding  Belgium ...
Geocoding  dope show ...
Geocoding  Oshawa, Canada ...
Geocoding  Baker City Oregon ...
Geocoding  United States ...
Geocoding  marysville ca  ...
Geocoding  Hermosa Beach, CA ...
Geocoding  19.600858, -99.047821 ...
Geocoding  Pennsylvania ...
Geocoding  Salt Lake City, Utah ...
Geocoding  Palo Alto, CA ...
Geocoding  Spain but Opa-Locka, FL ...
Geocoding  Jaipur, India ...
Geocoding  Hyderabad Telangana INDIA ...
Geocoding  Eagle Pass, Texas ...
Geocoding  bangalore ...
Geocoding  Financial News and Views ...


GeocoderUnavailable: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Financial+News+and+Views&format=json&limit=1 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=1)"))

In [ ]:
from notebook.services.config import ConfigManager
c = ConfigManager()
c.update('notebook', {"CodeCell": {"cm_config": {"autoCloseBrackets": False}}})
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))